# WiDS Global Datathon 2026 — Survival Ensemble

In [1]:

import os
import gc
import json
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from catboost import CatBoostClassifier, CatBoostRegressor
import xgboost as xgb

FAST_MODE = False
THREADS = 1
EPS = 1e-9
HORIZONS = [12.0, 24.0, 48.0, 72.0]
INTERVALS = [(0.0, 12.0), (12.0, 24.0), (24.0, 48.0), (48.0, 72.0)]

if FAST_MODE:
    SEEDS = [11, 23]
    N_FOLDS = 4
else:
    SEEDS = [11, 23, 47, 59]
    N_FOLDS = 5

BASE_FEATURES = [
    'num_perimeters_0_5h','dt_first_last_0_5h','low_temporal_resolution_0_5h',
    'area_first_ha','area_growth_abs_0_5h','area_growth_rel_0_5h','area_growth_rate_ha_per_h',
    'log1p_area_first','log1p_growth','log_area_ratio_0_5h','relative_growth_0_5h',
    'radial_growth_m','radial_growth_rate_m_per_h','centroid_displacement_m','centroid_speed_m_per_h',
    'spread_bearing_deg','spread_bearing_sin','spread_bearing_cos',
    'dist_min_ci_0_5h','dist_std_ci_0_5h','dist_change_ci_0_5h','dist_slope_ci_0_5h',
    'closing_speed_m_per_h','closing_speed_abs_m_per_h','projected_advance_m','dist_accel_m_per_h2',
    'dist_fit_r2_0_5h','alignment_cos','alignment_abs','cross_track_component','along_track_speed',
    'event_start_hour','event_start_dayofweek','event_start_month'
]


def _safe_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) == 0 or len(b) == 0:
        return 0.0
    if np.nanstd(a) < 1e-12 or np.nanstd(b) < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def monotonicize(curve: np.ndarray) -> np.ndarray:
    curve = np.asarray(curve, float)
    curve = np.clip(curve, 0.0, 1.0)
    curve = np.maximum.accumulate(curve, axis=1)
    curve = np.clip(curve, 0.0, 1.0)
    return curve


def curve_to_risk(curve: np.ndarray) -> np.ndarray:
    curve = monotonicize(curve)
    m1 = curve[:, 0]
    m2 = np.clip(curve[:, 1] - curve[:, 0], 0.0, 1.0)
    m3 = np.clip(curve[:, 2] - curve[:, 1], 0.0, 1.0)
    m4 = np.clip(curve[:, 3] - curve[:, 2], 0.0, 1.0)
    surv72 = np.clip(1.0 - curve[:, 3], 0.0, 1.0)
    expected_time = 6.0 * m1 + 18.0 * m2 + 36.0 * m3 + 60.0 * m4 + 96.0 * surv72
    return -expected_time


def harrell_c_index(times, events, risk):
    times = np.asarray(times, float)
    events = np.asarray(events, int)
    risk = np.asarray(risk, float)
    concordant = 0.0
    ties = 0.0
    comparable = 0.0
    n = len(times)
    for i in range(n):
        ti, ei, ri = times[i], events[i], risk[i]
        for j in range(i + 1, n):
            tj, ej, rj = times[j], events[j], risk[j]
            if ei == 1 and ti < tj:
                comparable += 1.0
                if ri > rj:
                    concordant += 1.0
                elif ri == rj:
                    ties += 1.0
            elif ej == 1 and tj < ti:
                comparable += 1.0
                if rj > ri:
                    concordant += 1.0
                elif ri == rj:
                    ties += 1.0
    if comparable == 0:
        return 0.5
    return (concordant + 0.5 * ties) / comparable


def brier_at_horizon(times, events, preds, H):
    times = np.asarray(times, float)
    events = np.asarray(events, int)
    preds = np.asarray(preds, float)
    mask = ~((events == 0) & (times < H))
    y = ((events == 1) & (times <= H)).astype(float)
    if mask.sum() == 0:
        return 0.25
    return float(np.mean((preds[mask] - y[mask]) ** 2))


def hybrid_metric(times, events, curve):
    curve = monotonicize(np.asarray(curve, float))
    risk = curve_to_risk(curve)
    c_idx = harrell_c_index(times, events, risk)
    b24 = brier_at_horizon(times, events, curve[:, 1], 24.0)
    b48 = brier_at_horizon(times, events, curve[:, 2], 48.0)
    b72 = brier_at_horizon(times, events, curve[:, 3], 72.0)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1.0 - wb)


def km_curve(times, events, horizons=HORIZONS):
    times = np.asarray(times, float)
    events = np.asarray(events, int)
    uniq = np.sort(np.unique(times[events == 1]))
    surv_vals = []
    s = 1.0
    for t in uniq:
        d = np.sum((times == t) & (events == 1))
        r = np.sum(times >= t)
        if r > 0:
            s *= (1.0 - d / r)
        surv_vals.append((t, s))
    out = []
    for H in horizons:
        surv = 1.0
        for t, sv in surv_vals:
            if t <= H:
                surv = sv
            else:
                break
        out.append(1.0 - surv)
    return np.array(out, float)


def find_single_file(filename, required_cols=None, forbidden_cols=None):
    search_roots = ['/kaggle/input', '.']
    candidates = []
    for root in search_roots:
        candidates.extend(glob.glob(os.path.join(root, '**', filename), recursive=True))
    candidates = sorted(set(candidates))
    valid = []
    for path in candidates:
        try:
            probe = pd.read_csv(path, nrows=5)
            cols = set(probe.columns)
            if required_cols and not set(required_cols).issubset(cols):
                continue
            if forbidden_cols and any(c in cols for c in forbidden_cols):
                continue
            valid.append(path)
        except Exception:
            continue
    if not valid:
        raise FileNotFoundError(f'Could not locate {filename}')
    def _score(path):
        low = path.lower()
        score = 0.0
        if 'samplecsv_sub' in low:
            score += 1000.0
        if filename != 'sample_submission.csv' and 'submission' in low:
            score += 100.0
        if 'anonymous-contest-2' in low or 'wids' in low or 'globaldathon' in low:
            score -= 10.0
        score += len(low) / 10000.0
        return score
    valid = sorted(valid, key=_score)
    return valid[0]


def read_data():
    train_path = find_single_file(
        'train.csv',
        required_cols=['event_id', 'time_to_hit_hours', 'event'] + BASE_FEATURES,
        forbidden_cols=None,
    )
    test_path = find_single_file(
        'test.csv',
        required_cols=['event_id'] + BASE_FEATURES,
        forbidden_cols=['time_to_hit_hours', 'event'],
    )
    sample_path = None
    try:
        sample_path = find_single_file(
            'sample_submission.csv',
            required_cols=['event_id'],
            forbidden_cols=None,
        )
    except Exception:
        sample_path = None

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sample_sub = pd.read_csv(sample_path) if sample_path is not None else None

    for col in BASE_FEATURES + ['time_to_hit_hours', 'event']:
        if col in train_df.columns:
            train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
    for col in BASE_FEATURES:
        if col in test_df.columns:
            test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

    assert train_df['event_id'].is_unique, 'train event_id duplicated'
    assert test_df['event_id'].is_unique, 'test event_id duplicated'
    assert set(train_df['event'].dropna().unique()).issubset({0, 1}), 'event must be binary'
    return train_df, test_df, sample_sub, train_path, test_path, sample_path


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()
    for c in BASE_FEATURES:
        if c not in X.columns:
            raise KeyError(f'missing feature: {c}')
        X[c] = pd.to_numeric(X[c], errors='coerce')

    X['event_start_hour_sin'] = np.sin(2.0 * np.pi * X['event_start_hour'] / 24.0)
    X['event_start_hour_cos'] = np.cos(2.0 * np.pi * X['event_start_hour'] / 24.0)
    X['event_start_dayofweek_sin'] = np.sin(2.0 * np.pi * X['event_start_dayofweek'] / 7.0)
    X['event_start_dayofweek_cos'] = np.cos(2.0 * np.pi * X['event_start_dayofweek'] / 7.0)
    X['event_start_month_sin'] = np.sin(2.0 * np.pi * (X['event_start_month'] - 1.0) / 12.0)
    X['event_start_month_cos'] = np.cos(2.0 * np.pi * (X['event_start_month'] - 1.0) / 12.0)

    log_abs_cols = [
        'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h',
        'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m',
        'dist_accel_m_per_h2', 'centroid_displacement_m', 'centroid_speed_m_per_h',
        'radial_growth_m', 'radial_growth_rate_m_per_h', 'area_growth_abs_0_5h',
        'area_growth_rate_ha_per_h', 'along_track_speed', 'cross_track_component'
    ]
    for c in log_abs_cols:
        X[f'log1p_abs__{c}'] = np.log1p(np.abs(X[c]))

    dist = np.maximum(np.abs(X['dist_min_ci_0_5h'].astype(float)), 1.0)
    dist_std = np.maximum(np.abs(X['dist_std_ci_0_5h'].astype(float)), 1.0)
    closing = X['closing_speed_m_per_h'].astype(float)
    closing_abs = np.maximum(np.abs(X['closing_speed_abs_m_per_h'].astype(float)), 1.0)
    along = X['along_track_speed'].astype(float)
    cross = X['cross_track_component'].astype(float)
    radial = X['radial_growth_m'].astype(float)
    radial_rate = X['radial_growth_rate_m_per_h'].astype(float)
    centroid_speed = X['centroid_speed_m_per_h'].astype(float)
    area0 = np.maximum(X['area_first_ha'].astype(float), 0.0)
    growth_abs = X['area_growth_abs_0_5h'].astype(float)
    growth_rate = X['area_growth_rate_ha_per_h'].astype(float)
    dt = np.maximum(X['dt_first_last_0_5h'].astype(float), 0.05)
    nper = np.maximum(X['num_perimeters_0_5h'].astype(float), 1.0)
    align = X['alignment_abs'].astype(float)
    align_cos = X['alignment_cos'].astype(float)
    dchange = X['dist_change_ci_0_5h'].astype(float)
    dslope = X['dist_slope_ci_0_5h'].astype(float)
    proj = X['projected_advance_m'].astype(float)
    fitr2 = X['dist_fit_r2_0_5h'].astype(float)
    low_res = X['low_temporal_resolution_0_5h'].astype(float)

    X['signed_time_to_contact_h'] = np.where(closing > 1.0, dist / np.maximum(closing, 1.0), 999.0)
    X['abs_time_to_contact_h'] = dist / closing_abs
    X['distance_uncertainty_ratio'] = dist_std / dist
    X['advance_ratio'] = proj / dist
    X['radial_to_distance'] = radial / dist
    X['radialrate_to_distance'] = radial_rate / dist
    X['growth_to_distance'] = growth_abs / dist
    X['growthrate_to_distance'] = growth_rate / dist
    X['initial_area_to_distance'] = area0 / dist
    X['alignment_x_closing'] = align * closing
    X['alignment_x_along'] = align * along
    X['alignmentcos_x_closing'] = align_cos * closing
    X['centroid_x_alignment'] = centroid_speed * align
    X['radial_x_alignment'] = radial_rate * align
    X['cross_track_abs'] = np.abs(cross)
    X['closing_minus_cross'] = closing_abs - np.abs(cross)
    X['temporal_density'] = nper / dt
    X['temporal_span_log'] = np.log1p(dt)
    X['uncertain_x_dist'] = low_res * np.log1p(dist)
    X['uncertain_x_growth'] = low_res * np.log1p(np.abs(growth_abs))
    X['area_x_closing'] = X['log1p_area_first'].astype(float) * closing
    X['growth_x_closing'] = X['log1p_growth'].astype(float) * closing
    X['bearing_alignment'] = X['spread_bearing_cos'].astype(float) * align_cos
    X['distance_trend_combo'] = dchange + proj
    X['distance_slope_over_dist'] = dslope / dist
    X['dist_accel_over_speed'] = X['dist_accel_m_per_h2'].astype(float) / closing_abs
    X['r2_x_closing'] = fitr2 * closing
    X['r2_x_alignment'] = fitr2 * align
    X['closing_is_positive'] = (closing > 0.0).astype(int)
    X['dist_is_closing'] = (dchange < 0.0).astype(int)
    X['high_alignment_flag'] = (align > 0.75).astype(int)
    X['weekend_flag'] = (X['event_start_dayofweek'].astype(float) >= 5.0).astype(int)
    X['night_flag'] = ((X['event_start_hour'].astype(float) <= 5.0) | (X['event_start_hour'].astype(float) >= 20.0)).astype(int)
    X['summerish_flag'] = X['event_start_month'].isin([5, 6, 7, 8, 9]).astype(int)

    X['dist_over_area0'] = dist / np.maximum(area0, 1.0)
    X['dist_over_growth'] = dist / np.maximum(np.abs(growth_abs), 1.0)
    X['speed_balance'] = along / np.maximum(closing_abs, 1.0)
    X['cross_over_closing'] = np.abs(cross) / np.maximum(closing_abs, 1.0)
    X['centroid_over_radial'] = centroid_speed / np.maximum(np.abs(radial_rate), 1.0)
    X['area_growth_efficiency'] = growth_abs / np.maximum(area0, 1.0)
    X['radial_efficiency'] = radial / np.maximum(centroid_speed, 1.0)
    X['movement_efficiency'] = proj / np.maximum(centroid_speed, 1.0)
    X['close_then_align'] = np.maximum(closing, 0.0) * np.maximum(align, 0.0)
    X['close_then_r2'] = np.maximum(closing, 0.0) * np.maximum(fitr2, 0.0)
    X['potential_reach_24h_m'] = proj + 24.0 * np.maximum(closing, 0.0)
    X['potential_reach_48h_m'] = proj + 48.0 * np.maximum(closing, 0.0)
    X['potential_reach_ratio_24h'] = X['potential_reach_24h_m'] / dist
    X['potential_reach_ratio_48h'] = X['potential_reach_48h_m'] / dist

    X = X.replace([np.inf, -np.inf], np.nan)

    protected = {'event_id', '_is_train'}
    na_cols = [c for c in X.columns if c not in protected and X[c].isna().any()]
    for c in na_cols:
        X[f'na__{c}'] = X[c].isna().astype(int)

    return X


def build_joint_features(train_df, test_df):
    tr = train_df.drop(columns=[c for c in ['time_to_hit_hours', 'event'] if c in train_df.columns]).copy()
    te = test_df.copy()
    tr['_is_train'] = 1
    te['_is_train'] = 0
    combo = pd.concat([tr, te], axis=0, ignore_index=True, sort=False)
    combo = build_features(combo)
    feature_cols = [c for c in combo.columns if c not in ['event_id', '_is_train']]
    X_train = combo.loc[combo['_is_train'] == 1, feature_cols].reset_index(drop=True)
    X_test = combo.loc[combo['_is_train'] == 0, feature_cols].reset_index(drop=True)
    return X_train, X_test, feature_cols


def make_strata(df):
    event = pd.Series(df['event']).astype(int)
    t = pd.Series(df['time_to_hit_hours']).astype(float)
    rank = t.rank(method='first')
    try:
        q = pd.qcut(rank, q=5, labels=False, duplicates='drop').astype(str)
    except Exception:
        q = pd.Series(np.zeros(len(df), dtype=int)).astype(str)
    return event.astype(str) + '_' + q


def get_cv_splits(df):
    strata = make_strata(df)
    splits = []
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(df, strata)):
            splits.append((seed, fold, tr_idx, va_idx))
    return splits


def balanced_weights(y):
    y = np.asarray(y, int)
    pos = int(y.sum())
    neg = int(len(y) - pos)
    if pos == 0 or neg == 0:
        return np.ones(len(y), float)
    w = np.ones(len(y), float)
    w[y == 1] = neg / max(pos, 1)
    return w


def expand_intervals_for_training(X, times, events):
    rows = []
    ys = []
    interval_ids = []
    for i, (t, e) in enumerate(zip(times, events)):
        for k, (L, U) in enumerate(INTERVALS):
            if t <= L:
                break
            if e == 0 and t < U:
                break
            rows.append(i)
            ys.append(1 if (e == 1 and t <= U) else 0)
            interval_ids.append(k)
            if e == 1 and t <= U:
                break
    longX = X.iloc[rows].reset_index(drop=True).copy()
    longX['interval_id'] = interval_ids
    longX['interval_start'] = [INTERVALS[k][0] for k in interval_ids]
    longX['interval_end'] = [INTERVALS[k][1] for k in interval_ids]
    longX['interval_mid'] = [(INTERVALS[k][0] + INTERVALS[k][1]) / 2.0 for k in interval_ids]
    longX['interval_len'] = [INTERVALS[k][1] - INTERVALS[k][0] for k in interval_ids]
    return longX, np.asarray(ys, int)


def expand_intervals_for_prediction(X):
    rep = np.repeat(np.arange(len(X)), len(INTERVALS))
    interval_ids = np.tile(np.arange(len(INTERVALS)), len(X))
    longX = X.iloc[rep].reset_index(drop=True).copy()
    longX['interval_id'] = interval_ids
    longX['interval_start'] = [INTERVALS[k][0] for k in interval_ids]
    longX['interval_end'] = [INTERVALS[k][1] for k in interval_ids]
    longX['interval_mid'] = [(INTERVALS[k][0] + INTERVALS[k][1]) / 2.0 for k in interval_ids]
    longX['interval_len'] = [INTERVALS[k][1] - INTERVALS[k][0] for k in interval_ids]
    return longX


def hazards_to_curve(haz):
    haz = np.asarray(haz, float)
    haz = np.clip(haz, EPS, 1.0 - EPS)
    surv = np.cumprod(1.0 - haz, axis=1)
    curve = 1.0 - surv
    return monotonicize(curve)


def train_catboost_binary_with_valid(Xtr, ytr, Xva, yva, params):
    ytr = np.asarray(ytr, int)
    if len(np.unique(ytr)) < 2:
        p = float(np.mean(ytr))
        return np.full(len(Xva), p, float), {'const': p}
    sw = balanced_weights(ytr)
    model = CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='Logloss',
        iterations=params.get('iterations', 600),
        depth=params.get('depth', 5),
        learning_rate=params.get('learning_rate', 0.03),
        l2_leaf_reg=params.get('l2_leaf_reg', 8.0),
        random_strength=params.get('random_strength', 1.0),
        bagging_temperature=params.get('bagging_temperature', 0.5),
        border_count=params.get('border_count', 64),
        grow_policy=params.get('grow_policy', 'SymmetricTree'),
        bootstrap_type=params.get('bootstrap_type', 'Bayesian'),
        random_seed=params.get('random_seed', 42),
        thread_count=THREADS,
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(
        Xtr, ytr,
        sample_weight=sw,
        eval_set=(Xva, np.asarray(yva, int)),
        use_best_model=True,
        verbose=False
    )
    pred = model.predict_proba(Xva)[:, 1]
    return pred, model


def predict_catboost_binary(model, X):
    if isinstance(model, dict) and 'const' in model:
        return np.full(len(X), model['const'], float)
    return model.predict_proba(X)[:, 1]


def train_xgb_binary(Xtr, ytr, Xva, yva, params):
    ytr = np.asarray(ytr, int)
    if len(np.unique(ytr)) < 2:
        p = float(np.mean(ytr))
        return np.full(len(Xva), p, float), {'const': p}
    dtr = xgb.DMatrix(Xtr, label=ytr, weight=balanced_weights(ytr))
    dva = xgb.DMatrix(Xva, label=np.asarray(yva, int))
    xparams = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'eta': params.get('learning_rate', 0.03),
        'max_depth': params.get('max_depth', 4),
        'min_child_weight': params.get('min_child_weight', 6),
        'subsample': params.get('subsample', 0.85),
        'colsample_bytree': params.get('colsample_bytree', 0.85),
        'lambda': params.get('reg_lambda', 2.0),
        'alpha': params.get('reg_alpha', 0.0),
        'gamma': params.get('gamma', 0.0),
        'max_bin': params.get('max_bin', 256),
        'verbosity': 0,
        'seed': params.get('random_seed', 42),
        'nthread': THREADS,
    }
    bst = xgb.train(
        xparams,
        dtr,
        num_boost_round=params.get('num_boost_round', 700),
        evals=[(dva, 'valid')],
        early_stopping_rounds=params.get('early_stopping_rounds', 80),
        verbose_eval=False,
    )
    pred = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1))
    return pred, bst


def predict_xgb_binary(model, X):
    if isinstance(model, dict) and 'const' in model:
        return np.full(len(X), model['const'], float)
    d = xgb.DMatrix(X)
    best_iter = getattr(model, 'best_iteration', None)
    if best_iter is None:
        return model.predict(d)
    return model.predict(d, iteration_range=(0, best_iter + 1))


def train_lr_binary(Xtr, ytr, Xva, params, long_mode=False):
    ytr = np.asarray(ytr, int)
    if len(np.unique(ytr)) < 2:
        p = float(np.mean(ytr))
        return np.full(len(Xva), p, float), {'const': p}
    if long_mode:
        cat_cols = ['interval_id']
        num_cols = [c for c in Xtr.columns if c not in cat_cols]
        pre = ColumnTransformer([
            ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ])
    else:
        pre = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())])
    lr = LogisticRegression(C=params.get('C', 1.0), max_iter=4000, solver='lbfgs', class_weight='balanced')
    pipe = Pipeline([('pre', pre), ('lr', lr)])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict_proba(Xva)[:, 1]
    return pred, pipe


def predict_lr_binary(model, X):
    if isinstance(model, dict) and 'const' in model:
        return np.full(len(X), model['const'], float)
    return model.predict_proba(X)[:, 1]


def train_et_binary(Xtr, ytr, Xva, params):
    ytr = np.asarray(ytr, int)
    if len(np.unique(ytr)) < 2:
        p = float(np.mean(ytr))
        return np.full(len(Xva), p, float), {'const': p}
    imp = SimpleImputer(strategy='median')
    Xtr_imp = imp.fit_transform(Xtr)
    Xva_imp = imp.transform(Xva)
    model = ExtraTreesClassifier(
        n_estimators=params.get('n_estimators', 600),
        max_depth=params.get('max_depth', None),
        min_samples_leaf=params.get('min_samples_leaf', 3),
        max_features=params.get('max_features', 'sqrt'),
        class_weight='balanced_subsample',
        random_state=params.get('random_seed', 42),
        n_jobs=THREADS,
    )
    model.fit(Xtr_imp, ytr)
    pred = model.predict_proba(Xva_imp)[:, 1]
    return pred, (imp, model)


def predict_et_binary(model, X):
    if isinstance(model, dict) and 'const' in model:
        return np.full(len(X), model['const'], float)
    imp, est = model
    return est.predict_proba(imp.transform(X))[:, 1]


def calibrate_curve_platt(oof_curve, test_curve, times, events):
    oof_curve = np.asarray(oof_curve, float)
    test_curve = np.asarray(test_curve, float)
    out_oof = np.zeros_like(oof_curve)
    out_test = np.zeros_like(test_curve)
    for j, H in enumerate(HORIZONS):
        mask = ~((events == 0) & (times < H))
        y = ((events == 1) & (times <= H)).astype(int)[mask]
        p = np.clip(oof_curve[mask, j], 1e-6, 1.0 - 1e-6)
        if len(np.unique(y)) < 2:
            const = float(np.mean(y)) if len(y) else 0.0
            out_oof[:, j] = const
            out_test[:, j] = const
            continue
        z = logit(p)
        Xcal = np.c_[z, z ** 2]
        lr = LogisticRegression(C=1.0, max_iter=4000)
        lr.fit(Xcal, y)
        all_z = logit(np.clip(oof_curve[:, j], 1e-6, 1.0 - 1e-6))
        test_z = logit(np.clip(test_curve[:, j], 1e-6, 1.0 - 1e-6))
        out_oof[:, j] = lr.predict_proba(np.c_[all_z, all_z ** 2])[:, 1]
        out_test[:, j] = lr.predict_proba(np.c_[test_z, test_z ** 2])[:, 1]
    return monotonicize(out_oof), monotonicize(out_test)


def calibrate_score_to_curve(oof_score, test_score, times, events):
    oof_score = np.asarray(oof_score, float)
    test_score = np.asarray(test_score, float)
    if harrell_c_index(times, events, oof_score) < harrell_c_index(times, events, -oof_score):
        oof_score = -oof_score
        test_score = -test_score
    out_oof = np.zeros((len(oof_score), len(HORIZONS)), float)
    out_test = np.zeros((len(test_score), len(HORIZONS)), float)
    for j, H in enumerate(HORIZONS):
        mask = ~((events == 0) & (times < H))
        y = ((events == 1) & (times <= H)).astype(int)[mask]
        x = oof_score[mask]
        if len(np.unique(y)) < 2 or np.nanstd(x) < 1e-12:
            const = float(np.mean(y)) if len(y) else 0.0
            out_oof[:, j] = const
            out_test[:, j] = const
            continue
        mu = float(np.mean(x))
        sd = float(np.std(x) + 1e-9)
        z = (x - mu) / sd
        Xcal = np.c_[z, z ** 2]
        lr = LogisticRegression(C=1.0, max_iter=4000)
        lr.fit(Xcal, y)
        all_z = (oof_score - mu) / sd
        test_z = (test_score - mu) / sd
        out_oof[:, j] = lr.predict_proba(np.c_[all_z, all_z ** 2])[:, 1]
        out_test[:, j] = lr.predict_proba(np.c_[test_z, test_z ** 2])[:, 1]
    return monotonicize(out_oof), monotonicize(out_test)


def run_pooled_hazard_model(train_df, test_df, X_train, X_test, family='catboost', params=None):
    params = params or {}
    n_train = len(train_df)
    n_test = len(test_df)
    oof_curve = np.zeros((n_train, 4), float)
    oof_count = np.zeros(n_train, float)
    test_curve = np.zeros((n_test, 4), float)
    splits = get_cv_splits(train_df)

    for seed, fold, tr_idx, va_idx in splits:
        Xtr = X_train.iloc[tr_idx].reset_index(drop=True)
        Xva = X_train.iloc[va_idx].reset_index(drop=True)
        ttr = train_df['time_to_hit_hours'].values[tr_idx]
        etr = train_df['event'].values[tr_idx]

        long_tr, y_tr = expand_intervals_for_training(Xtr, ttr, etr)
        valid_long, y_valid_long = expand_intervals_for_training(
            Xva,
            train_df['time_to_hit_hours'].values[va_idx],
            train_df['event'].values[va_idx],
        )
        pred_va_long = expand_intervals_for_prediction(Xva)
        pred_te_long = expand_intervals_for_prediction(X_test)

        if family == 'catboost':
            long_tr = long_tr.copy()
            valid_long = valid_long.copy()
            pred_va_long = pred_va_long.copy()
            pred_te_long = pred_te_long.copy()
            long_tr['interval_id'] = long_tr['interval_id'].astype(str)
            valid_long['interval_id'] = valid_long['interval_id'].astype(str)
            pred_va_long['interval_id'] = pred_va_long['interval_id'].astype(str)
            pred_te_long['interval_id'] = pred_te_long['interval_id'].astype(str)
            _, model = train_catboost_binary_with_valid(
                long_tr, y_tr, valid_long, y_valid_long,
                {**params, 'random_seed': seed * 100 + fold + 7}
            )
            pred_va_h = predict_catboost_binary(model, pred_va_long)
            pred_te_h = predict_catboost_binary(model, pred_te_long)
        elif family == 'xgb':
            _, model = train_xgb_binary(
                long_tr, y_tr, valid_long, y_valid_long,
                {**params, 'random_seed': seed * 100 + fold + 7}
            )
            pred_va_h = predict_xgb_binary(model, pred_va_long)
            pred_te_h = predict_xgb_binary(model, pred_te_long)
        elif family == 'lr':
            pred_va_h, model = train_lr_binary(long_tr, y_tr, pred_va_long, {**params}, long_mode=True)
            pred_te_h = predict_lr_binary(model, pred_te_long)
        elif family == 'et':
            pred_va_h, model = train_et_binary(long_tr, y_tr, pred_va_long, {**params, 'random_seed': seed * 100 + fold + 7})
            pred_te_h = predict_et_binary(model, pred_te_long)
        else:
            raise ValueError(f'unknown family: {family}')

        va_curve = hazards_to_curve(pred_va_h.reshape(len(va_idx), 4))
        te_curve = hazards_to_curve(pred_te_h.reshape(len(X_test), 4))

        oof_curve[va_idx] += va_curve
        oof_count[va_idx] += 1.0
        test_curve += te_curve / len(splits)

    oof_curve /= np.maximum(oof_count[:, None], 1.0)
    return monotonicize(oof_curve), monotonicize(test_curve)


def run_direct_horizon_model(train_df, test_df, X_train, X_test, family='catboost', params=None):
    params = params or {}
    n_train = len(train_df)
    n_test = len(test_df)
    oof_curve = np.zeros((n_train, 4), float)
    oof_count = np.zeros((n_train, 4), float)
    test_curve = np.zeros((n_test, 4), float)

    times = train_df['time_to_hit_hours'].values.astype(float)
    events = train_df['event'].values.astype(int)
    splits = get_cv_splits(train_df)

    for j, H in enumerate(HORIZONS):
        for seed, fold, tr_idx, va_idx in splits:
            tr_mask = ~((events[tr_idx] == 0) & (times[tr_idx] < H))
            Xtr = X_train.iloc[tr_idx].iloc[tr_mask].reset_index(drop=True)
            y_tr = ((events[tr_idx] == 1) & (times[tr_idx] <= H)).astype(int)[tr_mask]
            Xva = X_train.iloc[va_idx].reset_index(drop=True)
            y_va = ((events[va_idx] == 1) & (times[va_idx] <= H)).astype(int)

            if family == 'catboost':
                pred_va, model = train_catboost_binary_with_valid(
                    Xtr, y_tr, Xva, y_va,
                    {**params, 'random_seed': seed * 100 + fold + j + 17}
                )
                pred_te = predict_catboost_binary(model, X_test)
            elif family == 'xgb':
                pred_va, model = train_xgb_binary(
                    Xtr, y_tr, Xva, y_va,
                    {**params, 'random_seed': seed * 100 + fold + j + 17}
                )
                pred_te = predict_xgb_binary(model, X_test)
            elif family == 'lr':
                pred_va, model = train_lr_binary(Xtr, y_tr, Xva, {**params}, long_mode=False)
                pred_te = predict_lr_binary(model, X_test)
            elif family == 'et':
                pred_va, model = train_et_binary(
                    Xtr, y_tr, Xva,
                    {**params, 'random_seed': seed * 100 + fold + j + 17}
                )
                pred_te = predict_et_binary(model, X_test)
            else:
                raise ValueError(f'unknown family: {family}')

            oof_curve[va_idx, j] += pred_va
            oof_count[va_idx, j] += 1.0
            test_curve[:, j] += pred_te / len(splits)

    oof_curve /= np.maximum(oof_count, 1.0)
    return monotonicize(oof_curve), monotonicize(test_curve)


def run_survival_scalar_model(train_df, test_df, X_train, X_test, family='xgb_aft', params=None):
    params = params or {}
    n_train = len(train_df)
    n_test = len(test_df)
    oof_score = np.zeros(n_train, float)
    oof_count = np.zeros(n_train, float)
    test_score = np.zeros(n_test, float)

    times = train_df['time_to_hit_hours'].values.astype(float)
    events = train_df['event'].values.astype(int)
    splits = get_cv_splits(train_df)

    for seed, fold, tr_idx, va_idx in splits:
        Xtr = X_train.iloc[tr_idx]
        Xva = X_train.iloc[va_idx]
        ttr = times[tr_idx]
        etr = events[tr_idx]

        if family == 'xgb_cox':
            dtr = xgb.DMatrix(Xtr, label=np.where(etr == 1, ttr, -ttr))
            dva = xgb.DMatrix(Xva, label=np.where(events[va_idx] == 1, times[va_idx], -times[va_idx]))
            dte = xgb.DMatrix(X_test)
            xparams = {
                'objective': 'survival:cox',
                'eval_metric': 'cox-nloglik',
                'tree_method': 'hist',
                'eta': params.get('learning_rate', 0.03),
                'max_depth': params.get('max_depth', 3),
                'min_child_weight': params.get('min_child_weight', 8),
                'subsample': params.get('subsample', 0.85),
                'colsample_bytree': params.get('colsample_bytree', 0.85),
                'lambda': params.get('reg_lambda', 2.0),
                'alpha': params.get('reg_alpha', 0.0),
                'gamma': params.get('gamma', 0.0),
                'max_bin': params.get('max_bin', 256),
                'verbosity': 0,
                'seed': seed * 100 + fold + 31,
                'nthread': THREADS,
            }
            bst = xgb.train(
                xparams,
                dtr,
                num_boost_round=params.get('num_boost_round', 700),
                evals=[(dva, 'valid')],
                early_stopping_rounds=params.get('early_stopping_rounds', 80),
                verbose_eval=False,
            )
            va_score = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1))
            te_score = bst.predict(dte, iteration_range=(0, bst.best_iteration + 1))

        elif family == 'xgb_aft':
            dtr = xgb.DMatrix(Xtr)
            dtr.set_float_info('label_lower_bound', ttr)
            dtr.set_float_info('label_upper_bound', np.where(etr == 1, ttr, np.inf))
            dva = xgb.DMatrix(Xva)
            dva.set_float_info('label_lower_bound', times[va_idx])
            dva.set_float_info('label_upper_bound', np.where(events[va_idx] == 1, times[va_idx], np.inf))
            dte = xgb.DMatrix(X_test)
            xparams = {
                'objective': 'survival:aft',
                'eval_metric': 'aft-nloglik',
                'aft_loss_distribution': params.get('aft_loss_distribution', 'normal'),
                'aft_loss_distribution_scale': params.get('aft_loss_distribution_scale', 1.0),
                'tree_method': 'hist',
                'eta': params.get('learning_rate', 0.03),
                'max_depth': params.get('max_depth', 3),
                'min_child_weight': params.get('min_child_weight', 6),
                'subsample': params.get('subsample', 0.85),
                'colsample_bytree': params.get('colsample_bytree', 0.85),
                'lambda': params.get('reg_lambda', 2.0),
                'alpha': params.get('reg_alpha', 0.0),
                'gamma': params.get('gamma', 0.0),
                'max_bin': params.get('max_bin', 256),
                'verbosity': 0,
                'seed': seed * 100 + fold + 41,
                'nthread': THREADS,
            }
            bst = xgb.train(
                xparams,
                dtr,
                num_boost_round=params.get('num_boost_round', 700),
                evals=[(dva, 'valid')],
                early_stopping_rounds=params.get('early_stopping_rounds', 80),
                verbose_eval=False,
            )
            va_score = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1))
            te_score = bst.predict(dte, iteration_range=(0, bst.best_iteration + 1))

        elif family == 'cb_cox':
            ytr = np.where(etr == 1, ttr, -ttr)
            yva = np.where(events[va_idx] == 1, times[va_idx], -times[va_idx])
            model = CatBoostRegressor(
                loss_function='Cox',
                eval_metric='Cox',
                iterations=params.get('iterations', 600),
                depth=params.get('depth', 5),
                learning_rate=params.get('learning_rate', 0.03),
                l2_leaf_reg=params.get('l2_leaf_reg', 8.0),
                random_strength=params.get('random_strength', 1.0),
                bagging_temperature=params.get('bagging_temperature', 0.5),
                border_count=params.get('border_count', 64),
                bootstrap_type=params.get('bootstrap_type', 'Bayesian'),
                random_seed=seed * 100 + fold + 51,
                thread_count=THREADS,
                allow_writing_files=False,
                verbose=False,
            )
            model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True, verbose=False)
            va_score = model.predict(Xva)
            te_score = model.predict(X_test)

        elif family == 'cb_aft':
            ytr = np.c_[ttr, np.where(etr == 1, ttr, -1)]
            yva = np.c_[times[va_idx], np.where(events[va_idx] == 1, times[va_idx], -1)]
            loss = f"SurvivalAft:dist={params.get('dist', 'Normal')};scale={params.get('scale', 1.0)}"
            model = CatBoostRegressor(
                loss_function=loss,
                eval_metric=loss,
                iterations=params.get('iterations', 600),
                depth=params.get('depth', 5),
                learning_rate=params.get('learning_rate', 0.03),
                l2_leaf_reg=params.get('l2_leaf_reg', 8.0),
                random_strength=params.get('random_strength', 1.0),
                bagging_temperature=params.get('bagging_temperature', 0.5),
                border_count=params.get('border_count', 64),
                bootstrap_type=params.get('bootstrap_type', 'Bayesian'),
                random_seed=seed * 100 + fold + 61,
                thread_count=THREADS,
                allow_writing_files=False,
                verbose=False,
            )
            model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True, verbose=False)
            va_score = model.predict(Xva)
            te_score = model.predict(X_test)

        else:
            raise ValueError(f'unknown survival family: {family}')

        oof_score[va_idx] += va_score
        oof_count[va_idx] += 1.0
        test_score += te_score / len(splits)

    oof_score /= np.maximum(oof_count, 1.0)
    return oof_score, test_score


def build_candidate_models(train_df, test_df, X_train, X_test):
    times = train_df['time_to_hit_hours'].values.astype(float)
    events = train_df['event'].values.astype(int)
    candidates = {}

    km = km_curve(times, events)
    candidates['km'] = {
        'oof': np.tile(km, (len(train_df), 1)),
        'test': np.tile(km, (len(test_df), 1)),
    }

    model_specs = [
        ('hazard_cb', 'curve', lambda: run_pooled_hazard_model(train_df, test_df, X_train, X_test, 'catboost',
            {'iterations': 500, 'depth': 5, 'learning_rate': 0.035, 'l2_leaf_reg': 10.0, 'bagging_temperature': 0.8})),
        ('hazard_xgb', 'curve', lambda: run_pooled_hazard_model(train_df, test_df, X_train, X_test, 'xgb',
            {'num_boost_round': 600, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 4, 'subsample': 0.85, 'colsample_bytree': 0.85})),
        ('hazard_lr', 'curve', lambda: run_pooled_hazard_model(train_df, test_df, X_train, X_test, 'lr', {'C': 0.7})),

        ('direct_cb', 'curve', lambda: run_direct_horizon_model(train_df, test_df, X_train, X_test, 'catboost',
            {'iterations': 500, 'depth': 5, 'learning_rate': 0.035, 'l2_leaf_reg': 10.0, 'bagging_temperature': 0.8})),
        ('direct_xgb', 'curve', lambda: run_direct_horizon_model(train_df, test_df, X_train, X_test, 'xgb',
            {'num_boost_round': 600, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 4, 'subsample': 0.85, 'colsample_bytree': 0.85})),
        ('direct_lr', 'curve', lambda: run_direct_horizon_model(train_df, test_df, X_train, X_test, 'lr', {'C': 1.0})),
        ('direct_et', 'curve', lambda: run_direct_horizon_model(train_df, test_df, X_train, X_test, 'et',
            {'n_estimators': 500, 'max_depth': None, 'min_samples_leaf': 3, 'max_features': 'sqrt'})),

        ('cox_xgb', 'score', lambda: run_survival_scalar_model(train_df, test_df, X_train, X_test, 'xgb_cox',
            {'num_boost_round': 600, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 6})),
        ('cox_cb', 'score', lambda: run_survival_scalar_model(train_df, test_df, X_train, X_test, 'cb_cox',
            {'iterations': 500, 'depth': 5, 'learning_rate': 0.03, 'l2_leaf_reg': 10.0, 'bagging_temperature': 0.8})),

        ('aft_xgb_normal', 'score', lambda: run_survival_scalar_model(train_df, test_df, X_train, X_test, 'xgb_aft',
            {'aft_loss_distribution': 'normal', 'aft_loss_distribution_scale': 1.0, 'num_boost_round': 600, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 4})),
        ('aft_xgb_logistic', 'score', lambda: run_survival_scalar_model(train_df, test_df, X_train, X_test, 'xgb_aft',
            {'aft_loss_distribution': 'logistic', 'aft_loss_distribution_scale': 1.0, 'num_boost_round': 600, 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 4})),
        ('aft_cb_normal', 'score', lambda: run_survival_scalar_model(train_df, test_df, X_train, X_test, 'cb_aft',
            {'dist': 'Normal', 'scale': 1.0, 'iterations': 500, 'depth': 5, 'learning_rate': 0.03, 'l2_leaf_reg': 10.0, 'bagging_temperature': 0.8})),
        ('aft_cb_logistic', 'score', lambda: run_survival_scalar_model(train_df, test_df, X_train, X_test, 'cb_aft',
            {'dist': 'Logistic', 'scale': 1.0, 'iterations': 500, 'depth': 5, 'learning_rate': 0.03, 'l2_leaf_reg': 10.0, 'bagging_temperature': 0.8})),
    ]

    score_rows = []
    for name, mode, runner in model_specs:
        try:
            result = runner()
            if mode == 'curve':
                raw_oof, raw_test = result
                raw_oof = monotonicize(raw_oof)
                raw_test = monotonicize(raw_test)
                candidates[name] = {'oof': raw_oof, 'test': raw_test}
                score_rows.append({'model': name, 'score': hybrid_metric(times, events, raw_oof)})

                cal_oof, cal_test = calibrate_curve_platt(raw_oof, raw_test, times, events)
                candidates[f'{name}__platt'] = {'oof': cal_oof, 'test': cal_test}
                score_rows.append({'model': f'{name}__platt', 'score': hybrid_metric(times, events, cal_oof)})

            else:
                raw_oof_score, raw_test_score = result
                cal_oof, cal_test = calibrate_score_to_curve(raw_oof_score, raw_test_score, times, events)
                candidates[name] = {'oof': cal_oof, 'test': cal_test}
                score_rows.append({'model': name, 'score': hybrid_metric(times, events, cal_oof)})

        except Exception as e:
            print(f'SKIP {name}: {repr(e)}')
        gc.collect()

    score_df = pd.DataFrame(score_rows).sort_values('score', ascending=False).reset_index(drop=True)
    return candidates, score_df


def prune_candidates(candidates, times, events, max_keep=14, corr_threshold=0.997):
    scores = {
        name: hybrid_metric(times, events, payload['oof'])
        for name, payload in candidates.items()
    }
    ranked = sorted(scores, key=scores.get, reverse=True)
    keep = []
    keep_risks = []

    for name in ranked:
        risk = curve_to_risk(candidates[name]['oof'])
        if not keep:
            keep.append(name)
            keep_risks.append(risk)
            continue
        max_corr = max(abs(_safe_corr(risk, r)) for r in keep_risks)
        if max_corr < corr_threshold or len(keep) < 5:
            keep.append(name)
            keep_risks.append(risk)
        if len(keep) >= max_keep:
            break

    if 'km' in candidates and 'km' not in keep:
        keep.append('km')

    return keep, scores


def weighted_blend_from_names(names, weights, candidates):
    oof = np.zeros_like(candidates[names[0]]['oof'])
    test = np.zeros_like(candidates[names[0]]['test'])
    for w, n in zip(weights, names):
        oof += w * candidates[n]['oof']
        test += w * candidates[n]['test']
    return monotonicize(oof), monotonicize(test)


def greedy_blend(selected_names, candidates, times, events, max_rounds=40, alphas=None):
    if alphas is None:
        alphas = np.linspace(0.05, 0.95, 19)
    best_name = max(selected_names, key=lambda n: hybrid_metric(times, events, candidates[n]['oof']))
    blend_oof = candidates[best_name]['oof'].copy()
    blend_test = candidates[best_name]['test'].copy()
    weights = {best_name: 1.0}
    best_score = hybrid_metric(times, events, blend_oof)

    for _ in range(max_rounds):
        improved = False
        best_trial = None
        for name in selected_names:
            cand_oof = candidates[name]['oof']
            cand_test = candidates[name]['test']
            for a in alphas:
                trial_oof = monotonicize(a * blend_oof + (1.0 - a) * cand_oof)
                trial_score = hybrid_metric(times, events, trial_oof)
                if trial_score > best_score + 1e-6:
                    best_trial = (name, a, trial_oof, monotonicize(a * blend_test + (1.0 - a) * cand_test), trial_score)
                    best_score = trial_score
                    improved = True
        if not improved:
            break
        name, a, blend_oof, blend_test, _ = best_trial
        for k in list(weights.keys()):
            weights[k] *= a
        weights[name] = weights.get(name, 0.0) + (1.0 - a)

    s = sum(weights.values())
    if s > 0:
        for k in list(weights.keys()):
            weights[k] /= s

    ordered = sorted(weights)
    ordered_weights = np.array([weights[n] for n in ordered], float)
    return blend_oof, blend_test, ordered, ordered_weights, best_score


def coordinate_refine(names, init_weights, candidates, times, events):
    weights = np.asarray(init_weights, float).copy()
    weights = np.maximum(weights, 0.0)
    weights = weights / np.maximum(weights.sum(), EPS)
    best_oof, best_test = weighted_blend_from_names(names, weights, candidates)
    best_score = hybrid_metric(times, events, best_oof)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01]:
        improved = True
        while improved:
            improved = False
            for i in range(len(names)):
                for j in range(len(names)):
                    if i == j or weights[j] < step:
                        continue
                    trial = weights.copy()
                    trial[i] += step
                    trial[j] -= step
                    trial = np.maximum(trial, 0.0)
                    trial /= np.maximum(trial.sum(), EPS)
                    trial_oof, trial_test = weighted_blend_from_names(names, trial, candidates)
                    score = hybrid_metric(times, events, trial_oof)
                    if score > best_score + 1e-6:
                        weights = trial
                        best_oof, best_test = trial_oof, trial_test
                        best_score = score
                        improved = True
    return best_oof, best_test, weights, best_score


def horizonwise_stack(selected_names, candidates, train_df):
    times = train_df['time_to_hit_hours'].values.astype(float)
    events = train_df['event'].values.astype(int)

    Xoof = np.stack([candidates[n]['oof'] for n in selected_names], axis=2)
    Xtest = np.stack([candidates[n]['test'] for n in selected_names], axis=2)

    out_oof = np.zeros((Xoof.shape[0], len(HORIZONS)), float)
    out_test = np.zeros((Xtest.shape[0], len(HORIZONS)), float)

    for j, H in enumerate(HORIZONS):
        mask = ~((events == 0) & (times < H))
        y = ((events == 1) & (times <= H)).astype(int)[mask]

        train_feats = logit(np.clip(Xoof[mask, j, :], 1e-6, 1.0 - 1e-6))
        all_feats = logit(np.clip(Xoof[:, j, :], 1e-6, 1.0 - 1e-6))
        test_feats = logit(np.clip(Xtest[:, j, :], 1e-6, 1.0 - 1e-6))

        if len(np.unique(y)) < 2:
            const = float(np.mean(y)) if len(y) else 0.0
            out_oof[:, j] = const
            out_test[:, j] = const
            continue

        lr = LogisticRegression(C=0.25, max_iter=4000)
        lr.fit(train_feats, y)
        out_oof[:, j] = lr.predict_proba(all_feats)[:, 1]
        out_test[:, j] = lr.predict_proba(test_feats)[:, 1]

    return monotonicize(out_oof), monotonicize(out_test)


def make_final_blend(candidates, train_df):
    times = train_df['time_to_hit_hours'].values.astype(float)
    events = train_df['event'].values.astype(int)

    selected_names, single_scores = prune_candidates(candidates, times, events, max_keep=14, corr_threshold=0.997)

    greedy_oof, greedy_test, greedy_names, greedy_weights, greedy_score = greedy_blend(
        selected_names, candidates, times, events
    )

    refined_oof, refined_test, refined_weights, refined_score = coordinate_refine(
        greedy_names, greedy_weights, candidates, times, events
    )

    stack_oof, stack_test = horizonwise_stack(selected_names, candidates, train_df)
    stack_score = hybrid_metric(times, events, stack_oof)

    avg_oof = monotonicize(0.5 * refined_oof + 0.5 * stack_oof)
    avg_test = monotonicize(0.5 * refined_test + 0.5 * stack_test)
    avg_score = hybrid_metric(times, events, avg_oof)

    avg_platt_oof, avg_platt_test = calibrate_curve_platt(avg_oof, avg_test, times, events)
    avg_platt_score = hybrid_metric(times, events, avg_platt_oof)

    ensemble_bank = {
        'refined_greedy': (refined_oof, refined_test, refined_score),
        'stack': (stack_oof, stack_test, stack_score),
        'avg': (avg_oof, avg_test, avg_score),
        'avg_platt': (avg_platt_oof, avg_platt_test, avg_platt_score),
    }
    best_name = max(ensemble_bank, key=lambda k: ensemble_bank[k][2])
    best_oof, best_test, best_score = ensemble_bank[best_name]

    blend_weights = {name: float(w) for name, w in zip(greedy_names, refined_weights)}
    meta = {
        'selected_single_models': selected_names,
        'single_scores': {k: float(v) for k, v in single_scores.items()},
        'greedy_names': greedy_names,
        'refined_weights': blend_weights,
        'ensemble_scores': {k: float(v[2]) for k, v in ensemble_bank.items()},
        'chosen_ensemble': best_name,
        'chosen_score': float(best_score),
    }
    return best_oof, best_test, meta


def validate_submission(sub, test_df):
    required = ['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']
    assert list(sub.columns) == required, f'Bad columns: {sub.columns.tolist()}'
    assert len(sub) == len(test_df), 'Submission row count mismatch'
    assert sub['event_id'].is_unique, 'Duplicate event_id in submission'
    assert set(sub['event_id']) == set(test_df['event_id']), 'Submission IDs do not match test IDs'
    probs = sub[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values
    assert np.isfinite(probs).all(), 'Non-finite probabilities found'
    assert ((probs >= 0.0) & (probs <= 1.0)).all(), 'Probabilities outside [0, 1]'
    assert (sub['prob_12h'] <= sub['prob_24h']).all(), 'Non-monotone 12->24'
    assert (sub['prob_24h'] <= sub['prob_48h']).all(), 'Non-monotone 24->48'
    assert (sub['prob_48h'] <= sub['prob_72h']).all(), 'Non-monotone 48->72'


def main():
    train_df, test_df, sample_sub, train_path, test_path, sample_path = read_data()
    print('train_path =', train_path)
    print('test_path =', test_path)
    print('sample_submission_path =', sample_path)
    print('train_shape =', train_df.shape)
    print('test_shape =', test_df.shape)

    X_train, X_test, feature_cols = build_joint_features(train_df, test_df)
    print('n_features =', len(feature_cols))

    candidates, score_df = build_candidate_models(train_df, test_df, X_train, X_test)
    print('n_candidates =', len(candidates))
    print(score_df.head(20).to_string(index=False))

    best_oof, best_test, meta = make_final_blend(candidates, train_df)
    print('chosen_ensemble =', meta['chosen_ensemble'])
    print('chosen_oof_score =', meta['chosen_score'])

    pred = monotonicize(best_test)
    pred_df = pd.DataFrame({
        'event_id': test_df['event_id'].values,
        'prob_12h': pred[:, 0],
        'prob_24h': pred[:, 1],
        'prob_48h': pred[:, 2],
        'prob_72h': pred[:, 3],
    })

    if sample_sub is not None and 'event_id' in sample_sub.columns:
        sub = sample_sub[['event_id']].merge(pred_df, on='event_id', how='left', validate='one_to_one')
    else:
        sub = pred_df.copy()

    validate_submission(sub, test_df)

    score_df.to_csv('oof_model_scores.csv', index=False)
    with open('blend_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    sub.to_csv('submission.csv', index=False)

    print(sub.head().to_string(index=False))
    print('Saved: submission.csv, oof_model_scores.csv, blend_meta.json')


main()


train_path = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv
test_path = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv
sample_submission_path = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/sample_submission.csv
train_shape = (221, 37)
test_shape = (95, 35)
n_features = 104
n_candidates = 21
            model    score
hazard_xgb__platt 0.969947
 hazard_cb__platt 0.969745
 direct_cb__platt 0.968804
direct_xgb__platt 0.968651
       direct_xgb 0.968343
 aft_xgb_logistic 0.967467
           cox_cb 0.967283
       hazard_xgb 0.967029
        direct_cb 0.966851
   aft_xgb_normal 0.965718
        hazard_cb 0.962040
    aft_cb_normal 0.959509
  aft_cb_logistic 0.958889
 direct_et__platt 0.946018
          cox_xgb 0.945708
 hazard_lr__platt 0.941245
        direct_et 0.936944
 direct_lr__platt 0.931647
        hazard_lr 0.931066
        direct_lr 0.900281
chosen_ensemble = avg_platt
chosen_oof_score = 0.9724786886335255
 event_id  prob_12h  prob_2